# Assignment 5
## Data Preprocessing

We will be using a dataframe created from *Income Dirty Data.csv*. Download the file from D2L. 

1. Import the following modules
    - `pandas`
    - `numpy`
    - `preprocessing` from `sklearn` (for bonus question)
    - `KNNImputer` from `sklearn.impute` (for bonus question)
2. Create your dataframe from the file using `pandas`

In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.impute import KNNImputer

df = pd.read_csv("Income Dirty Data.csv")
df.head()

,ID,sex,age,income,tax_15_pct
0,1,Women,21,147168.0,22075.20
1,2,Female,29,119595.0,17939.25
2,3,Female,56,87770.0,13165.50
3,4,NaN,21,54259.0,8138.85
4,5,Male,28,NaN,160230.00


3. Calculate and display the following information
    - Total number of NaN values for **each column**
    - Percentage of NaN values in the dataset 
    - Number of rows *without* any NaN values

In [2]:
# Total NaN values for each column
print("NaN values per column:")
print(df.isna().sum())

# Percentage of NaN values in the whole dataset (out of every cell)
nan_pct = df.isna().sum().sum() / df.size * 100
print(f"\nPercentage of NaN values in the dataset: {nan_pct:.2f}%")

# Number of rows that have no NaN values at all
complete_rows = df.dropna().shape[0]
print(f"Rows without any NaN values: {complete_rows} of {len(df)}")

NaN values per column:
ID              0
sex            88
age             0
income        109
tax_15_pct     93
dtype: int64

Percentage of NaN values in the dataset: 5.80%
Rows without any NaN values: 733 of 1000


Besides missing values (NaN), the dataset contains errors. We have the following rules to check:

* All employees are adults (18+ years old)
* All employees pay 15% of their income for the tax
* All employees make money; no income should be <= 0 
---
4. Calculate and display the percentage of the data that does **NOT** violate any **one** of the rules.

In [3]:
# A row violates a rule only when we can actually see the bad value.
# (A NaN is missing data, not a rule violation, so NaN comparisons count as "no violation".)
underage   = df["age"] < 18
bad_income = df["income"] <= 0
bad_tax    = (df["income"].notna() & df["tax_15_pct"].notna()
              & ~np.isclose(df["tax_15_pct"], df["income"] * 0.15))

print(f"Rows with age < 18:            {underage.sum()}")
print(f"Rows with income <= 0:         {bad_income.sum()}")
print(f"Rows where tax != 15% income:  {bad_tax.sum()}")

violates_any = underage | bad_income | bad_tax
valid_pct = (~violates_any).sum() / len(df) * 100
print(f"\nRows that do NOT violate any rule: {(~violates_any).sum()} of {len(df)}")
print(f"Percentage of data that does NOT violate any rule: {valid_pct:.2f}%")

Rows with age < 18:            98
Rows with income <= 0:         44
Rows where tax != 15% income:  144

Rows that do NOT violate any rule: 768 of 1000
Percentage of data that does NOT violate any rule: 76.80%


Now that we have determined the number of erroneous datapoints in our set, let's work on correcting it as best we can.

5. Replace non *Female*/*Male* values in the **Sex** column with either *Female* or *Male* (e.g., Women --> Female)

In [4]:
print("Before:")
print(df["sex"].value_counts(dropna=False))

df["sex"] = df["sex"].replace({
    "Women": "Female", "Woman": "Female",
    "Men": "Male", "Man": "Male",
})

print("\nAfter:")
print(df["sex"].value_counts(dropna=False))

Before:
sex
Male      363
Female    341
NaN        88
Women      57
Woman      57
Man        47
Men        47
Name: count, dtype: int64

After:
sex
Male      457
Female    455
NaN        88
Name: count, dtype: int64


6. Replace non-positive **Age** values with NaN (`numpy.NaN`)
7. Replace non-positive **Income** values with NaN (`numpy.NaN`)
8. Replace non-positive **Tax (15%)** values with NaN (`numpy.NaN`)

In [5]:
df.loc[df["age"] <= 0, "age"] = np.nan
df.loc[df["income"] <= 0, "income"] = np.nan
df.loc[df["tax_15_pct"] <= 0, "tax_15_pct"] = np.nan

print("NaN values per column after replacing non-positive values:")
print(df.isna().sum())

NaN values per column after replacing non-positive values:
ID              0
sex            88
age            98
income        153
tax_15_pct    103
dtype: int64


The following question is a bonus (+10) question, but I'd encourage you to give it a try!

9. Use machine learning (`KNNImputer`) to impute all missing values (replaces NaN values with the most predicted values)
    - Will need to use a scaler and convert the values in the **Sex** column to a numeric value for algorithm to work properly
    - Show some of the data prior to imputing, and after imputing

In [6]:
# Rows that have at least one missing value, so we can see the imputation work
missing_ids = df[df.isna().any(axis=1)].head(10).index
print("Before imputing:")
display(df.loc[missing_ids])

# Encode Sex as a number (Female = 0, Male = 1); NaN stays NaN so it gets imputed too
data = df.drop(columns=["ID"]).copy()
data["sex"] = data["sex"].map({"Female": 0, "Male": 1})

# Scale so that income (~100,000) doesn't swamp age (~40) in the distance calculation
scaler = preprocessing.MinMaxScaler()
scaled = scaler.fit_transform(data)

imputer = KNNImputer(n_neighbors=5)
imputed = imputer.fit_transform(scaled)

# Undo the scaling and rebuild the dataframe
imputed_df = pd.DataFrame(scaler.inverse_transform(imputed), columns=data.columns, index=data.index)
imputed_df["sex"] = imputed_df["sex"].round().map({0: "Female", 1: "Male"})
imputed_df["age"] = imputed_df["age"].round().astype(int)
imputed_df.insert(0, "ID", df["ID"])

print("After imputing:")
display(imputed_df.loc[missing_ids].round(2))
print(f"Remaining NaN values: {imputed_df.isna().sum().sum()}")

Before imputing:


,ID,sex,age,income,tax_15_pct
3,4,NaN,21.0,54259.0,8138.85
4,5,Male,28.0,NaN,160230.00
5,6,Female,NaN,128326.0,19248.90
6,7,Female,NaN,NaN,NaN
7,8,Female,24.0,NaN,11820.60
11,12,Male,55.0,NaN,9776.25
13,14,Male,47.0,85300.0,NaN
19,20,Female,NaN,89991.0,NaN
22,23,Male,NaN,NaN,22500.00
27,28,NaN,21.0,76019.0,11402.85


After imputing:


,ID,sex,age,income,tax_15_pct
3,4,Male,21,54259.0,8138.85
4,5,Male,28,87077.0,160230.00
5,6,Female,36,128326.0,19248.90
6,7,Female,40,90680.8,55756.65
7,8,Female,24,110944.4,11820.60
11,12,Male,55,87077.0,9776.25
13,14,Male,47,85300.0,12487.68
19,20,Female,44,89991.0,57968.06
22,23,Male,45,87295.6,22500.00
27,28,Male,21,76019.0,11402.85


Remaining NaN values: 0


10. In the empty `Markdown` cell below, explain why it is important to clean a dataset before calculating analytics about the data

Analytics are only as good as the data they are computed from ("garbage in, garbage out"). In this dataset, only 733 of 1,000 rows were complete and only about 77% of rows passed the three business rules. There were negative ages and incomes, underage "employees", tax values that were not 15% of income, and the same category written five different ways (*Female*, *Women*, *Woman*, *Male*, *Men*, *Man*).

If we calculated statistics before cleaning:
- **Wrong numbers:** impossible values such as a negative income or a tax of $1.36 million pull the mean and standard deviation far from the truth.
- **Wrong groups:** grouping by *sex* would produce six categories instead of two, splitting the same people into different buckets.
- **Errors or silent drops:** many functions and ML models fail on NaN values or quietly drop those rows, which shrinks and can bias the sample.
- **Bad decisions:** any conclusions or models built on dirty data inherit its errors, and nothing in the output warns you.

Cleaning first (standardizing labels, flagging impossible values as missing, then imputing) makes the results accurate, consistent, and trustworthy. For example, the remaining extreme `tax_15_pct` values show that more cleaning (such as recomputing tax as 15% of income) would further improve the imputation.

### Submission to D2L Dropbox
- Submit this `Jupyter` file to D2L, renamed as **Last_First_Assignment5.ipynb** 
    - Replace '**Last**' and '**First**' with your first and last name

- Include the link to your GitHub repository (the URL of your repo page, for
   example `https://github.com/yourname/csci-4047-work`).
### How to add my code to GitHub?

1. Stage your file (this tells Git which changes to include):

      `git add Last_First_Assignment5.ipynb`

   To stage everything (you might not want to stage everything though) in the folder instead, use `git add .`

3. Commit your changes (this saves a snapshot with a message):

       git commit -m "Add Assignment 5"

   The text in quotes is your commit message. Make it describe what you
   did.

4. Push your commit up to GitHub:

       git push -u origin main

   The `-u origin main` part is only needed the first push. After that,
   `git push` alone is enough.

**What each command does, briefly**

- `git add` picks which files to include in the next save.
- `git commit` saves a snapshot of those files on your computer, with a
  message describing the change.
- `git push` uploads your saved commits to GitHub so they appear online.